In [ ]:
# 1. Install dependencies
!apt-get update -y && apt-get install -y build-essential git cmake libboost-system-dev libprotobuf-dev protobuf-compiler libssl-dev libgoogle-perftools-dev

# 2. Clone marian-dev if not already present
!git clone https://github.com/marian-nmt/marian-dev.git /kaggle/working/marian-dev

# 3. Clean and build with warnings disabled
%cd /kaggle/working/marian-dev
!mkdir -p build

# 1. Patch the deprecated thrust inheritance in cumsum.cu
!sed -i 's/: public thrust::unary_function<int, int>//g' src/tensors/gpu/cumsum.cu

# 2. Re-configure build directory
%cd /kaggle/working/marian-dev/build
!rm -rf *

!cmake .. \
  -DCOMPILE_CUDA=ON \
  -DCOMPILE_CPU=ON \
  -DCMAKE_CXX_FLAGS="-Wno-error"

# 3. Compile Marian
!make -j$(nproc)

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]           
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease                         
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]        
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,919 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,058 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]     
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRe

In [2]:
!/kaggle/working/marian-dev/build/marian --version
!/kaggle/working/marian-dev/build/marian-decoder --version

v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
v1.12.43 c9f287db 2025-06-23 19:47:45 +0000


In [3]:
import os

# Define Marian paths
MARIAN_BUILD = "/kaggle/working/marian-dev/build"
MARIAN = f"{MARIAN_BUILD}/marian"
MARIAN_DECODER = f"{MARIAN_BUILD}/marian-decoder"
MARIAN_VOCAB = f"{MARIAN_BUILD}/marian-vocab"

# Install sacrebleu and sentencepiece for evaluation & tokenization
!pip install -q sacrebleu sentencepiece

# Create directories for models and vocabularies
!mkdir -p /kaggle/working/models/seq2seq
!mkdir -p /kaggle/working/models/transformer
!mkdir -p /kaggle/working/models/ensemble
!mkdir -p /kaggle/working/data
!mkdir -p /kaggle/working/output/seq2seq
!mkdir -p /kaggle/working/output/transformer
!mkdir -p /kaggle/working/output/ensemble
!mkdir -p /kaggle/working/config/seq2seq/hi2my
!mkdir -p /kaggle/working/config/seq2seq/my2hi
!mkdir -p /kaggle/working/config/transformer/hi2my
!mkdir -p /kaggle/working/config/transformer/my2hi
!mkdir -p /kaggle/working/config/ensemble/hi2my
!mkdir -p /kaggle/working/config/ensemble/my2hi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.6 MB/s eta 0:00:00


In [4]:
%cd /kaggle/working/data

!wget https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/test.hi
!wget https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/test.my
!wget https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/train.hi
!wget https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/train.my
!wget https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/valid.hi
!wget https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/valid.my
!wget https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak4all/refs/heads/main/python/sylbreak4all.py

/kaggle/working/data
--2026-10-15 14:23:27--  https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/test.hi
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24117 (24K) [text/plain]
Saving to: ‘test.hi’

test.hi             100%[===================>]  23.55K  --.-KB/s    in 0.003s  

2026-10-15 14:23:28 (8.84 MB/s) - ‘test.hi’ saved [24117/24117]

--2026-10-15 14:23:28--  https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/test.my
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting

In [5]:
import sentencepiece as spm
# Train Hieroglyph tokenizer
spm.SentencePieceTrainer.train(
  input='/kaggle/working/data/train.hi',
  model_prefix='hi_spm',
  vocab_size=4000,
  character_coverage=1.0,
  model_type='bpe'
)

# Train Burmese tokenizer
spm.SentencePieceTrainer.train(
  input='/kaggle/working/data/train.my',
  model_prefix='my_spm',
  vocab_size=8000,
  character_coverage=0.999,
  model_type='bpe'
)

I0000 00:00:1792074218.513300      49 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: /kaggle/working/data/train.hi
  input_format: 
  model_prefix: hi_spm
  model_type: BPE
  vocab_size: 4000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  ena

True

In [6]:
import sentencepiece as spm

# Initialize SentencePiece processors
hi_spm = spm.SentencePieceProcessor(model_file="hi_spm.model")
my_spm = spm.SentencePieceProcessor(model_file="my_spm.model")


def encode_file(input_path, output_path, sp_processor):
  with open(input_path, "r", encoding="utf-8") as f_in, open(
      output_path, "w", encoding="utf-8"
  ) as f_out:
    for line in f_in:
      pieces = sp_processor.encode(line.strip(), out_type=str)
      f_out.write(" ".join(pieces) + "\n")


# Process train, validation, and test splits
splits = ["train", "valid", "test"]
for split in splits:
  encode_file(
      f"/kaggle/working/data/{split}.hi",
      f"/kaggle/working/data/{split}.hi.spm",
      hi_spm,
  )
  encode_file(
      f"/kaggle/working/data/{split}.my",
      f"/kaggle/working/data/{split}.my.spm",
      my_spm,
  )

print("All datasets successfully encoded into subwords!")

All datasets successfully encoded into subwords!


In [7]:
# Generate vocabularies for source (Hieroglyph) and target (Burmese)
MARIAN_VOCAB="/kaggle/working/marian-dev/build/marian-vocab"

!{MARIAN_VOCAB} < /kaggle/working/data/train.hi.spm > /kaggle/working/data/hi.vocab.yml
!{MARIAN_VOCAB} < /kaggle/working/data/train.my.spm > /kaggle/working/data/my.vocab.yml

[2026-10-15 14:24:07] Creating vocabulary...
[2026-10-15 14:24:07] [data] Creating vocabulary stdout from stdin
[2026-10-15 14:24:07] Finished
[2026-10-15 14:24:08] Creating vocabulary...
[2026-10-15 14:24:08] [data] Creating vocabulary stdout from stdin
[2026-10-15 14:24:08] Finished


In [8]:
%%writefile /kaggle/working/config/transformer/hi2my/config.yml
# Device and Model Settings
devices: [0]
model: /kaggle/working/models/transformer/hi-my.npz
type: transformer

# Training Corpora
train-sets:
  - /kaggle/working/data/train.hi.spm
  - /kaggle/working/data/train.my.spm
vocabs:
  - /kaggle/working/data/hi.vocab.yml
  - /kaggle/working/data/my.vocab.yml

# Validation Settings
valid-sets:
  - /kaggle/working/data/valid.hi.spm
  - /kaggle/working/data/valid.my.spm
valid-freq: 100
valid-metrics: [cross-entropy,bleu]
early-stopping: 15

# Optimization & Hyperparameters (Corrected Marian Keys)
after-batches: 5000
mini-batch-words: 2048
workspace: 4096
optimizer: adam
learn-rate: 0.0003
lr-warmup: 4000
lr-decay-inv-sqrt: 4000
label-smoothing: 0.1

# Regularization
dropout-rnn: 0.2
dropout-src: 0.2
dropout-trg: 0.2
transformer-dropout: 0.2

# Corrected Architecture Parameters
transformer-heads: 4
dim-emb: 512
transformer-dim-ffn: 2048
enc-depth: 4
dec-depth: 4

Writing /kaggle/working/config/transformer/hi2my/config.yml


In [20]:
MARIAN_TRAIN="/kaggle/working/marian-dev/build/marian"
!{MARIAN_TRAIN} --config /kaggle/working/config/transformer/hi2my/config.yml

[2026-10-15 15:35:35] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 15:35:35] [marian] Running on f2fe76130532 as process 18643 with command line:
[2026-10-15 15:35:35] [marian] /kaggle/working/marian-dev/build/marian --config /kaggle/working/config/transformer/hi2my/config.yml
[2026-10-15 15:35:35] [config] after: 0e
[2026-10-15 15:35:35] [config] after-batches: 5000
[2026-10-15 15:35:35] [config] after-epochs: 0
[2026-10-15 15:35:35] [config] all-caps-every: 0
[2026-10-15 15:35:35] [config] allow-unk: false
[2026-10-15 15:35:35] [config] authors: false
[2026-10-15 15:35:35] [config] beam-size: 12
[2026-10-15 15:35:35] [config] bert-class-symbol: "[CLS]"
[2026-10-15 15:35:35] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 15:35:35] [config] bert-masking-fraction: 0.15
[2026-10-15 15:35:35] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 15:35:35] [config] bert-train-type-embeddings: true
[2026-10-15 15:35:35] [config] bert-type-vocab-size: 2
[2026-10-15 15:35:

In [21]:
MARIAN_DECODER="/kaggle/working/marian-dev/build/marian-decoder"

!{MARIAN_DECODER} \
  --models /kaggle/working/models/transformer/hi-my.npz \
  --vocabs /kaggle/working/data/hi.vocab.yml /kaggle/working/data/my.vocab.yml \
  --beam-size 12 \
  --normalize 1.0 \
  < /kaggle/working/data/test.hi.spm \
  > /kaggle/working/output/transformer/test.output.my.spm

[2026-10-15 15:44:49] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 15:44:49] [marian] Running on f2fe76130532 as process 18683 with command line:
[2026-10-15 15:44:49] [marian] /kaggle/working/marian-dev/build/marian-decoder --models /kaggle/working/models/transformer/hi-my.npz --vocabs /kaggle/working/data/hi.vocab.yml /kaggle/working/data/my.vocab.yml --beam-size 12 --normalize 1.0
[2026-10-15 15:44:49] [config] alignment: ""
[2026-10-15 15:44:49] [config] allow-special: false
[2026-10-15 15:44:49] [config] allow-unk: false
[2026-10-15 15:44:49] [config] authors: false
[2026-10-15 15:44:49] [config] beam-size: 12
[2026-10-15 15:44:49] [config] bert-class-symbol: "[CLS]"
[2026-10-15 15:44:49] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 15:44:49] [config] bert-masking-fraction: 0.15
[2026-10-15 15:44:49] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 15:44:49] [config] bert-train-type-embeddings: true
[2026-10-15 15:44:49] [config] bert-type-vocab-size: 2


In [22]:
import sentencepiece as spm

my_spm = spm.SentencePieceProcessor(model_file="my_spm.model")

with open(
    "/kaggle/working/output/transformer/test.output.my.spm", "r", encoding="utf-8"
) as f_in, open("/kaggle/working/output/transformer/test.output.my", "w", encoding="utf-8") as f_out:
  for line in f_in:
    pieces = line.strip().split()
    decoded_text = my_spm.decode(pieces)
    f_out.write(decoded_text + "\n")

print("Test set successfully detokenized!")

Test set successfully detokenized!


In [24]:
print("----Scoring Hieroglyph -> Burmese Transformer Model----")
!sacrebleu /kaggle/working/data/test.my -i /kaggle/working/output/transformer/test.output.my

----Scoring Hieroglyph -> Burmese Transformer Model----
{
 "name": "BLEU",
 "score": 15.0,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "31.0/14.6/9.1/12.5 (BP = 1.000 ratio = 1.026 hyp_len = 981 ref_len = 956)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}


In [25]:
%%writefile /kaggle/working/config/transformer/my2hi/config.yml
# Device and Model Settings
devices: [0]
model: /kaggle/working/models/transformer/my-hi.npz
type: transformer

# Training Corpora (Burmese -> Hieroglyph)
train-sets:
  - /kaggle/working/data/train.my.spm
  - /kaggle/working/data/train.hi.spm
vocabs:
  - /kaggle/working/data/my.vocab.yml
  - /kaggle/working/data/hi.vocab.yml

# Validation Settings
valid-sets:
  - /kaggle/working/data/valid.my.spm
  - /kaggle/working/data/valid.hi.spm
valid-freq: 100
valid-metrics: [cross-entropy, bleu]
early-stopping: 15

# Optimization & Hyperparameters
after-batches: 5000
mini-batch-words: 2048
workspace: 4096
optimizer: adam
learn-rate: 0.0003
lr-warmup: 4000
lr-decay-inv-sqrt: 4000
label-smoothing: 0.1

# Regularization
dropout-rnn: 0.2
dropout-src: 0.2
dropout-trg: 0.2
transformer-dropout: 0.2

# Architecture Parameters
transformer-heads: 4
dim-emb: 512
transformer-dim-ffn: 2048
enc-depth: 4
dec-depth: 4

Writing /kaggle/working/config/transformer/my2hi/config.yml


In [26]:
MARIAN_TRAIN = "/kaggle/working/marian-dev/build/marian"
!{MARIAN_TRAIN} --config /kaggle/working/config/transformer/my2hi/config.yml

[2026-10-15 15:49:55] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 15:49:55] [marian] Running on f2fe76130532 as process 18788 with command line:
[2026-10-15 15:49:55] [marian] /kaggle/working/marian-dev/build/marian --config /kaggle/working/config/transformer/my2hi/config.yml
[2026-10-15 15:49:55] [config] after: 0e
[2026-10-15 15:49:55] [config] after-batches: 5000
[2026-10-15 15:49:55] [config] after-epochs: 0
[2026-10-15 15:49:55] [config] all-caps-every: 0
[2026-10-15 15:49:55] [config] allow-unk: false
[2026-10-15 15:49:55] [config] authors: false
[2026-10-15 15:49:55] [config] beam-size: 12
[2026-10-15 15:49:55] [config] bert-class-symbol: "[CLS]"
[2026-10-15 15:49:55] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 15:49:55] [config] bert-masking-fraction: 0.15
[2026-10-15 15:49:55] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 15:49:55] [config] bert-train-type-embeddings: true
[2026-10-15 15:49:55] [config] bert-type-vocab-size: 2
[2026-10-15 15:49:

In [27]:
MARIAN_DECODER="/kaggle/working/marian-dev/build/marian-decoder"

!{MARIAN_DECODER} \
  --models /kaggle/working/models/transformer/my-hi.npz \
  --vocabs /kaggle/working/data/my.vocab.yml /kaggle/working/data/hi.vocab.yml \
  --beam-size 12 \
  --normalize 1.0 \
  < /kaggle/working/data/test.my.spm \
  > /kaggle/working/output/transformer/test.output.hi.spm

[2026-10-15 16:00:40] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 16:00:40] [marian] Running on f2fe76130532 as process 18828 with command line:
[2026-10-15 16:00:40] [marian] /kaggle/working/marian-dev/build/marian-decoder --models /kaggle/working/models/transformer/my-hi.npz --vocabs /kaggle/working/data/my.vocab.yml /kaggle/working/data/hi.vocab.yml --beam-size 12 --normalize 1.0
[2026-10-15 16:00:40] [config] alignment: ""
[2026-10-15 16:00:40] [config] allow-special: false
[2026-10-15 16:00:40] [config] allow-unk: false
[2026-10-15 16:00:40] [config] authors: false
[2026-10-15 16:00:40] [config] beam-size: 12
[2026-10-15 16:00:40] [config] bert-class-symbol: "[CLS]"
[2026-10-15 16:00:40] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 16:00:40] [config] bert-masking-fraction: 0.15
[2026-10-15 16:00:40] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 16:00:40] [config] bert-train-type-embeddings: true
[2026-10-15 16:00:40] [config] bert-type-vocab-size: 2


In [29]:
import sentencepiece as spm

hi_spm = spm.SentencePieceProcessor(model_file="hi_spm.model")

with open(
    "/kaggle/working/output/transformer/test.output.hi.spm", "r", encoding="utf-8"
) as f_in, open("/kaggle/working/output/transformer/test.output.hi", "w", encoding="utf-8") as f_out:
  for line in f_in:
    pieces = line.strip().split()
    decoded_text = my_spm.decode(pieces)
    f_out.write(decoded_text + "\n")

print("Test set successfully detokenized!")
print("----Scoring Burmese -> Hieroglyph Transformer Model")
!sacrebleu /kaggle/working/data/test.hi -i /kaggle/working/output/transformer/test.output.hi

Test set successfully detokenized!
----Scoring Burmese -> Hieroglyph Transformer Model
{
 "name": "BLEU",
 "score": 0.0,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "0.0/0.0/0.0/0.0 (BP = 0.117 ratio = 0.318 hyp_len = 500 ref_len = 1573)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}


In [30]:
!sacrebleu /kaggle/working/data/test.hi -i /kaggle/working/output/transformer/test.output.hi --tok char

{
 "name": "BLEU",
 "score": 20.8,
 "signature": "nrefs:1|case:mixed|eff:no|tok:char|smooth:exp|version:2.6.0",
 "verbose_score": "48.5/27.0/16.4/8.6 (BP = 1.000 ratio = 1.278 hyp_len = 7201 ref_len = 5636)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "char",
 "smooth": "exp",
 "version": "2.6.0"
}


In [31]:
!sacrebleu /kaggle/working/data/test.hi -i /kaggle/working/output/transformer/test.output.hi -m chrf

{
 "name": "chrF2",
 "score": 21.9,
 "signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0",
 "nrefs": "1",
 "case": "mixed",
 "eff": "yes",
 "nc": "6",
 "nw": "0",
 "space": "no",
 "version": "2.6.0"
}


In [35]:
import pandas as pd

# Define paths to your test dataset files
src_file = "/kaggle/working/data/test.my"
ref_file = "/kaggle/working/data/test.hi"
pred_file = "/kaggle/working/output/transformer/test.output.hi"

# Read lines from all three files
with open(src_file, "r", encoding="utf-8") as f_src, open(
    ref_file, "r", encoding="utf-8"
) as f_ref, open(pred_file, "r", encoding="utf-8") as f_pred:
  src_lines = [line.strip() for line in f_src]
  ref_lines = [line.strip() for line in f_ref]
  pred_lines = [line.strip() for line in f_pred]

# Select 10 sample indices to inspect
sample_indices = [0, 1, 2, 5, 10, 20, 40, 80, 160, 320]

samples = []
for idx in sample_indices:
  if idx < len(src_lines):
    samples.append({
        "Index": idx,
        "Source (Burmese)": src_lines[idx],
        "Reference (Hieroglyphs)": ref_lines[idx],
        "Prediction (Hieroglyphs)": pred_lines[idx],
    })

# Format as a clean Pandas DataFrame for easy viewing
df_samples = pd.DataFrame(samples)
pd.set_option("display.max_colwidth", None)
display(df_samples)

,Index,Source (Burmese),Reference (Hieroglyphs),Prediction (Hieroglyphs)
0,0,အချစ်က နာကျင်စေတယ်။,𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙,▁𓈎𓊪𓆑𓈙▁𓆑𓈙▁𓂨𓀀𓎡𓋴𓎛𓈙
1,1,ကျွန်မမှာ အစားအစာမရှိဘူး။,𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌,▁𓈎𓊪𓂝𓋴𓂝𓈙▁𓎜𓀀▁𓅲𓀀𓐍𓇌
2,2,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။,𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝,▁𓎜𓀀𓇋𓐍𓇌𓂧▁𓈙𓄿𓏏▁𓉔𓂝
3,5,ခရီးကောင်းပါ,𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡,▁𓊪𓄿𓎼𓂝▁𓄿▁𓈙𓄿𓅱𓂝▁𓈎𓋴𓆑𓄡
4,10,နင့်နံ့က ကောင်းတယ်,𓂨𓀀𓎡 𓈙𓉔𓂝𓂋𓂋 𓃀𓐍𓐍𓇌,▁𓆑𓈎▁𓆑𓈙▁𓇋𓐍𓋴𓆑𓎛𓃀
5,20,ဒါက ကြီးမားပါတယ်။,𓈎𓊪𓆑𓈙 𓆑𓈙 𓊪𓎡𓃀𓂝,▁𓈎𓊪𓆑𓈙▁𓆑𓈙▁𓇋𓆑𓃀
6,40,ကံမကောင်းခဲ့ဘူး။,𓆑𓈎 𓏏𓄿𓈙𓎛𓈎 𓂋𓎡𓏭𓈖,▁𓏏𓂝▁𓏏𓂝𓋴𓂝𓎛𓈎▁𓂋𓎡𓏭𓈖𓂧
7,80,သူတွေ့တယ်။,𓊪𓂝 𓅲𓀀𓎡𓎛𓇌 𓆑𓈎,▁𓊪𓂝▁𓏏𓄿𓈙▁𓐍𓎡𓈎𓋴𓄿𓃀𓂝𓇌
8,160,ပိုဆိုးလာတယ်,𓆑𓈎 𓃀𓂝𓈎𓈙 𓏏𓐍𓋴𓈙𓂝,▁𓆑𓈎▁𓇋𓂝𓃀𓄿𓎛▁𓈎𓐍▁𓉔𓂝
9,320,ကျွန်တော် အလုပ်ကြိုးစားတယ်။,𓆑𓂋𓂋 𓏏𓐍𓋴𓈖 𓊪𓄿𓋴𓇌,▁𓆑𓉔▁𓊪𓄿𓋴𓇌𓏏𓐍𓋴𓈖𓆑𓎛𓃀


In [36]:
import pandas as pd

# Define paths to your test dataset files
src_file = "/kaggle/working/data/test.hi"
ref_file = "/kaggle/working/data/test.my"
pred_file = "/kaggle/working/output/transformer/test.output.my"

# Read lines from all three files
with open(src_file, "r", encoding="utf-8") as f_src, open(
    ref_file, "r", encoding="utf-8"
) as f_ref, open(pred_file, "r", encoding="utf-8") as f_pred:
  src_lines = [line.strip() for line in f_src]
  ref_lines = [line.strip() for line in f_ref]
  pred_lines = [line.strip() for line in f_pred]

# Select 10 sample indices to inspect
sample_indices = [0, 1, 2, 5, 10, 20, 40, 80, 160, 320]

samples = []
for idx in sample_indices:
  if idx < len(src_lines):
    samples.append({
        "Index": idx,
        "Source (Hieroglyphs)": src_lines[idx],
        "Reference (Burmese)": ref_lines[idx],
        "Prediction (Burmese)": pred_lines[idx],
    })

# Format as a clean Pandas DataFrame for easy viewing
df_samples = pd.DataFrame(samples)
pd.set_option("display.max_colwidth", None)
display(df_samples)

,Index,Source (Hieroglyphs),Reference (Burmese),Prediction (Burmese)
0,0,𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙,အချစ်က နာကျင်စေတယ်။,နာကျင်တယ်
1,1,𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌,ကျွန်မမှာ အစားအစာမရှိဘူး။,ကျွန်မမှာ အစားအစာရှိတယ်
2,2,𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။
3,5,𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡,ခရီးကောင်းပါ,ခရီးကောင်းပါ
4,10,𓂨𓀀𓎡 𓈙𓉔𓂝𓂋𓂋 𓃀𓐍𓐍𓇌,နင့်နံ့က ကောင်းတယ်,နှင်းဆီတွေက အနံ့ကောင်းတယ်။
5,20,𓈎𓊪𓆑𓈙 𓆑𓈙 𓊪𓎡𓃀𓂝,ဒါက ကြီးမားပါတယ်။,ဒါက ခွေးတစ်ကောင်ပါ။
6,40,𓆑𓈎 𓏏𓄿𓈙𓎛𓈎 𓂋𓎡𓏭𓈖,ကံမကောင်းခဲ့ဘူး။,အဲဒါက ငါ့ဟာ မဟုတ်ခဲ့ဘူး။
7,80,𓊪𓂝 𓅲𓀀𓎡𓎛𓇌 𓆑𓈎,သူတွေ့တယ်။,သူက ကျွန်တော့်စက်ဘီးကိုတွေ့တယ်
8,160,𓆑𓈎 𓃀𓂝𓈎𓈙 𓏏𓐍𓋴𓈙𓂝,ပိုဆိုးလာတယ်,ဒါဟာ အမှန်ဖြစ်လောက်တယ်။
9,320,𓆑𓂋𓂋 𓏏𓐍𓋴𓈖 𓊪𓄿𓋴𓇌,ကျွန်တော် အလုပ်ကြိုးစားတယ်။,အလုပ်မကောင်းခြင်း


In [10]:
%%writefile /kaggle/working/config/seq2seq/hi2my/config.yml
devices: [0]
model: /kaggle/working/models/seq2seq/hi-my-rnn.npz
type: s2s

train-sets:
  - /kaggle/working/data/train.hi.spm
  - /kaggle/working/data/train.my.spm
vocabs:
  - /kaggle/working/data/hi.vocab.yml
  - /kaggle/working/data/my.vocab.yml

valid-sets:
  - /kaggle/working/data/valid.hi.spm
  - /kaggle/working/data/valid.my.spm
valid-freq: 100
valid-metrics: [cross-entropy, bleu]
early-stopping: 15

after-batches: 5000
mini-batch-words: 2048
workspace: 4096
optimizer: adam
learn-rate: 0.0003
lr-warmup: 500

# RNN Regularization & Architecture
enc-type: bidirectional
enc-cell: gru
dec-cell: gru
dropout-rnn: 0.2
dropout-src: 0.2
dropout-trg: 0.2
dim-emb: 512
dim-rnn: 1024
enc-depth: 1
dec-depth: 1

Overwriting /kaggle/working/config/seq2seq/hi2my/config.yml


In [11]:
%%writefile /kaggle/working/config/seq2seq/my2hi/config.yml
devices: [0]
model: /kaggle/working/models/seq2seq/my-hi-rnn.npz
type: s2s

train-sets:
  - /kaggle/working/data/train.my.spm
  - /kaggle/working/data/train.hi.spm
vocabs:
  - /kaggle/working/data/my.vocab.yml
  - /kaggle/working/data/hi.vocab.yml

valid-sets:
  - /kaggle/working/data/valid.my.spm
  - /kaggle/working/data/valid.hi.spm
valid-freq: 100
valid-metrics: [cross-entropy, bleu]
early-stopping: 15

after-batches: 5000
mini-batch-words: 2048
workspace: 4096
optimizer: adam
learn-rate: 0.0003
lr-warmup: 500

# RNN Regularization & Architecture
enc-type: bidirectional
enc-cell: gru
dec-cell: gru
dropout-rnn: 0.2
dropout-src: 0.2
dropout-trg: 0.2
dim-emb: 512
dim-rnn: 1024
enc-depth: 1
dec-depth: 1

Writing /kaggle/working/config/seq2seq/my2hi/config.yml


In [12]:
MARIAN_TRAIN = "/kaggle/working/marian-dev/build/marian"

print("--- Training Hieroglyph -> Burmese RNN Model ---")
!{MARIAN_TRAIN} --config /kaggle/working/config/seq2seq/hi2my/config.yml

--- Training Hieroglyph -> Burmese RNN Model ---
[2026-10-15 14:27:34] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 14:27:34] [marian] Running on f2fe76130532 as process 18376 with command line:
[2026-10-15 14:27:34] [marian] /kaggle/working/marian-dev/build/marian --config /kaggle/working/config/seq2seq/hi2my/config.yml
[2026-10-15 14:27:34] [config] after: 0e
[2026-10-15 14:27:34] [config] after-batches: 5000
[2026-10-15 14:27:34] [config] after-epochs: 0
[2026-10-15 14:27:34] [config] all-caps-every: 0
[2026-10-15 14:27:34] [config] allow-unk: false
[2026-10-15 14:27:34] [config] authors: false
[2026-10-15 14:27:34] [config] beam-size: 12
[2026-10-15 14:27:34] [config] bert-class-symbol: "[CLS]"
[2026-10-15 14:27:34] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 14:27:34] [config] bert-masking-fraction: 0.15
[2026-10-15 14:27:34] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 14:27:34] [config] bert-train-type-embeddings: true
[2026-10-15 14:27:34] [confi

In [13]:
MARIAN_TRAIN = "/kaggle/working/marian-dev/build/marian"

print("--- Training Burmese -> Hieroglyph RNN Model ---")
!{MARIAN_TRAIN} --config /kaggle/working/config/seq2seq/my2hi/config.yml

--- Training Burmese -> Hieroglyph RNN Model ---
[2026-10-15 14:40:25] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 14:40:25] [marian] Running on f2fe76130532 as process 18422 with command line:
[2026-10-15 14:40:25] [marian] /kaggle/working/marian-dev/build/marian --config /kaggle/working/config/seq2seq/my2hi/config.yml
[2026-10-15 14:40:25] [config] after: 0e
[2026-10-15 14:40:25] [config] after-batches: 5000
[2026-10-15 14:40:25] [config] after-epochs: 0
[2026-10-15 14:40:25] [config] all-caps-every: 0
[2026-10-15 14:40:25] [config] allow-unk: false
[2026-10-15 14:40:25] [config] authors: false
[2026-10-15 14:40:25] [config] beam-size: 12
[2026-10-15 14:40:25] [config] bert-class-symbol: "[CLS]"
[2026-10-15 14:40:25] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 14:40:25] [config] bert-masking-fraction: 0.15
[2026-10-15 14:40:25] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 14:40:25] [config] bert-train-type-embeddings: true
[2026-10-15 14:40:25] [confi

In [14]:
MARIAN_DECODER="/kaggle/working/marian-dev/build/marian-decoder"

!{MARIAN_DECODER} \
  --models /kaggle/working/models/seq2seq/hi-my-rnn.npz \
  --vocabs /kaggle/working/data/hi.vocab.yml /kaggle/working/data/my.vocab.yml \
  --beam-size 12 \
  --normalize 1.0 \
  < /kaggle/working/data/test.hi.spm \
  > /kaggle/working/output/seq2seq/test.output.my.spm

!{MARIAN_DECODER} \
  --models /kaggle/working/models/seq2seq/my-hi-rnn.npz \
  --vocabs /kaggle/working/data/my.vocab.yml /kaggle/working/data/hi.vocab.yml \
  --beam-size 12 \
  --normalize 1.0 \
  < /kaggle/working/data/test.my.spm \
  > /kaggle/working/output/seq2seq/test.output.hi.spm

[2026-10-15 15:06:23] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 15:06:23] [marian] Running on f2fe76130532 as process 18495 with command line:
[2026-10-15 15:06:23] [marian] /kaggle/working/marian-dev/build/marian-decoder --models /kaggle/working/models/seq2seq/hi-my-rnn.npz --vocabs /kaggle/working/data/hi.vocab.yml /kaggle/working/data/my.vocab.yml --beam-size 12 --normalize 1.0
[2026-10-15 15:06:24] [config] alignment: ""
[2026-10-15 15:06:24] [config] allow-special: false
[2026-10-15 15:06:24] [config] allow-unk: false
[2026-10-15 15:06:24] [config] authors: false
[2026-10-15 15:06:24] [config] beam-size: 12
[2026-10-15 15:06:24] [config] bert-class-symbol: "[CLS]"
[2026-10-15 15:06:24] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 15:06:24] [config] bert-masking-fraction: 0.15
[2026-10-15 15:06:24] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 15:06:24] [config] bert-train-type-embeddings: true
[2026-10-15 15:06:24] [config] bert-type-vocab-size: 2


In [15]:
import sentencepiece as spm

my_spm = spm.SentencePieceProcessor(model_file="my_spm.model")
hi_spm = spm.SentencePieceProcessor(model_file="hi_spm.model")

with open(
    "/kaggle/working/output/seq2seq/test.output.my.spm", "r", encoding="utf-8"
) as f_in, open("/kaggle/working/output/seq2seq/test.output.my", "w", encoding="utf-8") as f_out:
  for line in f_in:
    pieces = line.strip().split()
    decoded_text = my_spm.decode(pieces)
    f_out.write(decoded_text + "\n")

with open(
    "/kaggle/working/output/seq2seq/test.output.hi.spm", "r", encoding="utf-8"
) as f_in, open("/kaggle/working/output/seq2seq/test.output.hi", "w", encoding="utf-8") as f_out:
  for line in f_in:
    pieces = line.strip().split()
    decoded_text = hi_spm.decode(pieces)
    f_out.write(decoded_text + "\n")

print("Test set successfully detokenized!")

Test set successfully detokenized!


In [17]:
print("--- Scoring Hieroglyph -> Burmese RNN Model ---")
!sacrebleu /kaggle/working/data/test.my -i /kaggle/working/output/seq2seq/test.output.my
print("--- Scoring Burmese -> Hieroglyph RNN Model ---")
!sacrebleu /kaggle/working/data/test.hi -i /kaggle/working/output/seq2seq/test.output.hi

--- Scoring Hieroglyph -> Burmese RNN Model ---
{
 "name": "BLEU",
 "score": 19.8,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "27.7/13.6/12.4/33.3 (BP = 1.000 ratio = 1.024 hyp_len = 979 ref_len = 956)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}
--- Scoring Burmese -> Hieroglyph RNN Model ---
{
 "name": "BLEU",
 "score": 11.8,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "39.3/17.7/7.7/3.7 (BP = 1.000 ratio = 1.067 hyp_len = 1678 ref_len = 1573)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}


In [18]:
import pandas as pd

# Define paths to your test dataset files
src_file = "/kaggle/working/data/test.my"
ref_file = "/kaggle/working/data/test.hi"
pred_file = "/kaggle/working/output/seq2seq/test.output.hi"

# Read lines from all three files
with open(src_file, "r", encoding="utf-8") as f_src, open(
    ref_file, "r", encoding="utf-8"
) as f_ref, open(pred_file, "r", encoding="utf-8") as f_pred:
  src_lines = [line.strip() for line in f_src]
  ref_lines = [line.strip() for line in f_ref]
  pred_lines = [line.strip() for line in f_pred]

# Select 10 sample indices to inspect
sample_indices = [0, 1, 2, 5, 10, 20, 40, 80, 160, 320]

samples = []
for idx in sample_indices:
  if idx < len(src_lines):
    samples.append({
        "Index": idx,
        "Source (Burmese)": src_lines[idx],
        "Reference (Hieroglyphs)": ref_lines[idx],
        "Prediction (Hieroglyphs)": pred_lines[idx],
    })

# Format as a clean Pandas DataFrame for easy viewing
df_samples = pd.DataFrame(samples)
pd.set_option("display.max_colwidth", None)
display(df_samples)

,Index,Source (Burmese),Reference (Hieroglyphs),Prediction (Hieroglyphs)
0,0,အချစ်က နာကျင်စေတယ်။,𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙,𓈎𓊪𓆑𓈙 𓆑𓈙 𓉔𓂧 𓆑𓇌𓂝𓄿
1,1,ကျွန်မမှာ အစားအစာမရှိဘူး။,𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌,𓆑 𓊪𓄿𓇌 𓎜𓀀 𓅲𓀀𓐍𓇌
2,2,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။,𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝,𓎜𓀀𓇋𓐍𓇌𓂧 𓈙𓄿𓏏 𓉔𓂝
3,5,ခရီးကောင်းပါ,𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡,𓊪𓄿𓎼𓂝 𓄿 𓈙𓄿𓅱𓂝 𓈎𓋴𓆑𓄡
4,10,နင့်နံ့က ကောင်းတယ်,𓂨𓀀𓎡 𓈙𓉔𓂝𓂋𓂋 𓃀𓐍𓐍𓇌,𓆑𓈙 𓈎𓐍𓉔 𓄿𓎛 𓂝𓂋𓈖
5,20,ဒါက ကြီးမားပါတယ်။,𓈎𓊪𓆑𓈙 𓆑𓈙 𓊪𓎡𓃀𓂝,𓈎𓊪𓆑𓈙 𓆑𓈙 𓇋𓆑𓃀
6,40,ကံမကောင်းခဲ့ဘူး။,𓆑𓈎 𓏏𓄿𓈙𓎛𓈎 𓂋𓎡𓏭𓈖,𓏏𓂝 𓏏𓂝𓋴𓂝𓎛𓈎 𓂋𓎡𓏭𓈖𓂧
7,80,သူတွေ့တယ်။,𓊪𓂝 𓅲𓀀𓎡𓎛𓇌 𓆑𓈎,𓊪𓂝 𓇌𓆑𓇌 𓆑𓈎 𓄿𓃀𓄿𓆑𓎛
8,160,ပိုဆိုးလာတယ်,𓆑𓈎 𓃀𓂝𓈎𓈙 𓏏𓐍𓋴𓈙𓂝,𓆑𓉔 𓈙𓎡𓏭𓊪 𓄿 𓅲𓀀𓐍𓂋
9,320,ကျွန်တော် အလုပ်ကြိုးစားတယ်။,𓆑𓂋𓂋 𓏏𓐍𓋴𓈖 𓊪𓄿𓋴𓇌,𓆑𓉔 𓊪𓄿𓋴𓇌𓏏𓐍𓋴𓈖𓆑𓎛𓃀


In [19]:
import pandas as pd

# Define paths to your test dataset files
src_file = "/kaggle/working/data/test.hi"
ref_file = "/kaggle/working/data/test.my"
pred_file = "/kaggle/working/output/seq2seq/test.output.my"

# Read lines from all three files
with open(src_file, "r", encoding="utf-8") as f_src, open(
    ref_file, "r", encoding="utf-8"
) as f_ref, open(pred_file, "r", encoding="utf-8") as f_pred:
  src_lines = [line.strip() for line in f_src]
  ref_lines = [line.strip() for line in f_ref]
  pred_lines = [line.strip() for line in f_pred]

# Select 10 sample indices to inspect
sample_indices = [0, 1, 2, 5, 10, 20, 40, 80, 160, 320]

samples = []
for idx in sample_indices:
  if idx < len(src_lines):
    samples.append({
        "Index": idx,
        "Source (Hieroglyphs)": src_lines[idx],
        "Reference (Burmese)": ref_lines[idx],
        "Prediction (Burmese)": pred_lines[idx],
    })

# Format as a clean Pandas DataFrame for easy viewing
df_samples = pd.DataFrame(samples)
pd.set_option("display.max_colwidth", None)
display(df_samples)

,Index,Source (Hieroglyphs),Reference (Burmese),Prediction (Burmese)
0,0,𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙,အချစ်က နာကျင်စေတယ်။,ကျွန်မ ပြောနေတာ
1,1,𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌,ကျွန်မမှာ အစားအစာမရှိဘူး။,ကျွန်မမှာ အစားအစာရှိတယ်
2,2,𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။
3,5,𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡,ခရီးကောင်းပါ,ခရီးကောင်းပါ
4,10,𓂨𓀀𓎡 𓈙𓉔𓂝𓂋𓂋 𓃀𓐍𓐍𓇌,နင့်နံ့က ကောင်းတယ်,မင်းက မကောင်းဘူး
5,20,𓈎𓊪𓆑𓈙 𓆑𓈙 𓊪𓎡𓃀𓂝,ဒါက ကြီးမားပါတယ်။,ဒါက ခွေးတစ်ကောင်ပါ။
6,40,𓆑𓈎 𓏏𓄿𓈙𓎛𓈎 𓂋𓎡𓏭𓈖,ကံမကောင်းခဲ့ဘူး။,ဆူညံသံတွေ အများကြီးပါ။
7,80,𓊪𓂝 𓅲𓀀𓎡𓎛𓇌 𓆑𓈎,သူတွေ့တယ်။,သူအောင်မြင်ခဲ့တယ်။
8,160,𓆑𓈎 𓃀𓂝𓈎𓈙 𓏏𓐍𓋴𓈙𓂝,ပိုဆိုးလာတယ်,ဒါဟာ အဓိပ္ပါယ်မရှိပါဘူး။
9,320,𓆑𓂋𓂋 𓏏𓐍𓋴𓈖 𓊪𓄿𓋴𓇌,ကျွန်တော် အလုပ်ကြိုးစားတယ်။,Tom က ငိုလိမ့်မယ်။


In [32]:
MARIAN_DECODER="/kaggle/working/marian-dev/build/marian-decoder"

!{MARIAN_DECODER} \
  --models /kaggle/working/models/transformer/hi-my.npz /kaggle/working/models/seq2seq/hi-my-rnn.npz \
  --weights 0.6 0.4 \
  --vocabs /kaggle/working/data/hi.vocab.yml /kaggle/working/data/my.vocab.yml \
  --beam-size 12 \
  --normalize 1.0 \
  < /kaggle/working/data/test.hi.spm \
  > /kaggle/working/output/ensemble/test.output.my.ensemble.spm

[2026-10-15 16:28:06] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 16:28:06] [marian] Running on f2fe76130532 as process 18916 with command line:
[2026-10-15 16:28:06] [marian] /kaggle/working/marian-dev/build/marian-decoder --models /kaggle/working/models/transformer/hi-my.npz /kaggle/working/models/seq2seq/hi-my-rnn.npz --weights 0.6 0.4 --vocabs /kaggle/working/data/hi.vocab.yml /kaggle/working/data/my.vocab.yml --beam-size 12 --normalize 1.0
[2026-10-15 16:28:07] [config] alignment: ""
[2026-10-15 16:28:07] [config] allow-special: false
[2026-10-15 16:28:07] [config] allow-unk: false
[2026-10-15 16:28:07] [config] authors: false
[2026-10-15 16:28:07] [config] beam-size: 12
[2026-10-15 16:28:07] [config] bert-class-symbol: "[CLS]"
[2026-10-15 16:28:07] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 16:28:07] [config] bert-masking-fraction: 0.15
[2026-10-15 16:28:07] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 16:28:07] [config] bert-train-type-embedding

In [33]:
!{MARIAN_DECODER} \
  --models /kaggle/working/models/transformer/my-hi.npz /kaggle/working/models/seq2seq/my-hi-rnn.npz \
  --weights 0.6 0.4 \
  --vocabs /kaggle/working/data/my.vocab.yml /kaggle/working/data/hi.vocab.yml \
  --beam-size 12 \
  --normalize 1.0 \
  < /kaggle/working/data/test.my.spm \
  > /kaggle/working/output/ensemble/test.output.hi.ensemble.spm

[2026-10-15 16:34:54] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-15 16:34:54] [marian] Running on f2fe76130532 as process 18937 with command line:
[2026-10-15 16:34:54] [marian] /kaggle/working/marian-dev/build/marian-decoder --models /kaggle/working/models/transformer/my-hi.npz /kaggle/working/models/seq2seq/my-hi-rnn.npz --weights 0.6 0.4 --vocabs /kaggle/working/data/my.vocab.yml /kaggle/working/data/hi.vocab.yml --beam-size 12 --normalize 1.0
[2026-10-15 16:34:54] [config] alignment: ""
[2026-10-15 16:34:54] [config] allow-special: false
[2026-10-15 16:34:54] [config] allow-unk: false
[2026-10-15 16:34:54] [config] authors: false
[2026-10-15 16:34:54] [config] beam-size: 12
[2026-10-15 16:34:54] [config] bert-class-symbol: "[CLS]"
[2026-10-15 16:34:54] [config] bert-mask-symbol: "[MASK]"
[2026-10-15 16:34:54] [config] bert-masking-fraction: 0.15
[2026-10-15 16:34:54] [config] bert-sep-symbol: "[SEP]"
[2026-10-15 16:34:54] [config] bert-train-type-embedding

In [34]:
import sentencepiece as spm

my_spm = spm.SentencePieceProcessor(model_file="my_spm.model")
hi_spm = spm.SentencePieceProcessor(model_file="hi_spm.model")

with open(
    "/kaggle/working/output/ensemble/test.output.my.ensemble.spm", "r", encoding="utf-8"
) as f_in, open("/kaggle/working/output/ensemble/test.output.my", "w", encoding="utf-8") as f_out:
  for line in f_in:
    pieces = line.strip().split()
    decoded_text = my_spm.decode(pieces)
    f_out.write(decoded_text + "\n")

with open(
    "/kaggle/working/output/ensemble/test.output.hi.ensemble.spm", "r", encoding="utf-8"
) as f_in, open("/kaggle/working/output/ensemble/test.output.hi", "w", encoding="utf-8") as f_out:
  for line in f_in:
    pieces = line.strip().split()
    decoded_text = hi_spm.decode(pieces)
    f_out.write(decoded_text + "\n")

print("Test set successfully detokenized!")

Test set successfully detokenized!


In [36]:
print("----Scoring Burmese -> Hieroglyph Ensemble Model----")
!sacrebleu /kaggle/working/data/test.my -i /kaggle/working/output/ensemble/test.output.my

print("----Scoring Hieroglyph -> Burmese Ensemble Model----")
!sacrebleu /kaggle/working/data/test.hi -i /kaggle/working/output/ensemble/test.output.hi

----Scoring Burmese -> Hieroglyph Ensemble Model----
{
 "name": "BLEU",
 "score": 19.1,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "32.0/15.7/11.4/23.1 (BP = 1.000 ratio = 1.044 hyp_len = 998 ref_len = 956)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}
----Scoring Hieroglyph -> Burmese Ensemble Model----
{
 "name": "BLEU",
 "score": 17.2,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "45.4/21.8/11.3/7.8 (BP = 1.000 ratio = 1.041 hyp_len = 1638 ref_len = 1573)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}


In [37]:
import pandas as pd

# Define paths to your test dataset files
src_file = "/kaggle/working/data/test.my"
ref_file = "/kaggle/working/data/test.hi"
pred_file = "/kaggle/working/output/ensemble/test.output.hi"

# Read lines from all three files
with open(src_file, "r", encoding="utf-8") as f_src, open(
    ref_file, "r", encoding="utf-8"
) as f_ref, open(pred_file, "r", encoding="utf-8") as f_pred:
  src_lines = [line.strip() for line in f_src]
  ref_lines = [line.strip() for line in f_ref]
  pred_lines = [line.strip() for line in f_pred]

# Select 10 sample indices to inspect
sample_indices = [0, 1, 2, 5, 10, 20, 40, 80, 160, 320]

samples = []
for idx in sample_indices:
  if idx < len(src_lines):
    samples.append({
        "Index": idx,
        "Source (Burmese)": src_lines[idx],
        "Reference (Hieroglyphs)": ref_lines[idx],
        "Prediction (Hieroglyphs)": pred_lines[idx],
    })

# Format as a clean Pandas DataFrame for easy viewing
df_samples = pd.DataFrame(samples)
pd.set_option("display.max_colwidth", None)
display(df_samples)

,Index,Source (Burmese),Reference (Hieroglyphs),Prediction (Hieroglyphs)
0,0,အချစ်က နာကျင်စေတယ်။,𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙,𓈎𓊪𓆑𓈙 𓆑𓈙 𓎜𓀀𓈎 𓂋𓐍𓎼𓂝
1,1,ကျွန်မမှာ အစားအစာမရှိဘူး။,𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌,𓆑 𓊪𓄿𓇌 𓎜𓀀 𓅲𓀀𓐍𓇌
2,2,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။,𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝,𓎜𓀀𓇋𓐍𓇌𓂧 𓈙𓄿𓏏 𓉔𓂝
3,5,ခရီးကောင်းပါ,𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡,𓊪𓄿𓎼𓂝 𓄿 𓈙𓄿𓅱𓂝 𓈎𓋴𓆑𓄡
4,10,နင့်နံ့က ကောင်းတယ်,𓂨𓀀𓎡 𓈙𓉔𓂝𓂋𓂋 𓃀𓐍𓐍𓇌,𓆑𓈙 𓈎𓐍𓉔 𓂨𓀀𓎡𓋴 𓊪𓄿𓎛𓇌
5,20,ဒါက ကြီးမားပါတယ်။,𓈎𓊪𓆑𓈙 𓆑𓈙 𓊪𓎡𓃀𓂝,𓈎𓊪𓆑𓈙 𓆑𓈙 𓇋𓆑𓃀
6,40,ကံမကောင်းခဲ့ဘူး။,𓆑𓈎 𓏏𓄿𓈙𓎛𓈎 𓂋𓎡𓏭𓈖,𓏏𓂝 𓏏𓂝𓋴𓂝𓎛𓈎 𓂋𓎡𓏭𓈖𓂧
7,80,သူတွေ့တယ်။,𓊪𓂝 𓅲𓀀𓎡𓎛𓇌 𓆑𓈎,𓊪𓂝 𓇌𓋴𓐍𓄡𓄡𓂝𓇌 𓆑𓈎
8,160,ပိုဆိုးလာတယ်,𓆑𓈎 𓃀𓂝𓈎𓈙 𓏏𓐍𓋴𓈙𓂝,𓆑𓉔 𓎜𓀀𓈎 𓂝𓎼𓆑𓂋
9,320,ကျွန်တော် အလုပ်ကြိုးစားတယ်။,𓆑𓂋𓂋 𓏏𓐍𓋴𓈖 𓊪𓄿𓋴𓇌,𓆑𓉔 𓊪𓄿𓋴𓇌𓏏𓐍𓋴𓈖𓆑𓎛𓃀


In [38]:
import pandas as pd

# Define paths to your test dataset files
src_file = "/kaggle/working/data/test.hi"
ref_file = "/kaggle/working/data/test.my"
pred_file = "/kaggle/working/output/ensemble/test.output.my"

# Read lines from all three files
with open(src_file, "r", encoding="utf-8") as f_src, open(
    ref_file, "r", encoding="utf-8"
) as f_ref, open(pred_file, "r", encoding="utf-8") as f_pred:
  src_lines = [line.strip() for line in f_src]
  ref_lines = [line.strip() for line in f_ref]
  pred_lines = [line.strip() for line in f_pred]

# Select 10 sample indices to inspect
sample_indices = [0, 1, 2, 5, 10, 20, 40, 80, 160, 320]

samples = []
for idx in sample_indices:
  if idx < len(src_lines):
    samples.append({
        "Index": idx,
        "Source (Hieroglyphs)": src_lines[idx],
        "Reference (Burmese)": ref_lines[idx],
        "Prediction (Burmese)": pred_lines[idx],
    })

# Format as a clean Pandas DataFrame for easy viewing
df_samples = pd.DataFrame(samples)
pd.set_option("display.max_colwidth", None)
display(df_samples)

,Index,Source (Hieroglyphs),Reference (Burmese),Prediction (Burmese)
0,0,𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙,အချစ်က နာကျင်စေတယ်။,ကျွန်မ နိုင်တာကို ကြိုက်တယ်။
1,1,𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌,ကျွန်မမှာ အစားအစာမရှိဘူး။,ကျွန်မမှာ အစားအစာရှိတယ်
2,2,𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။,ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။
3,5,𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡,ခရီးကောင်းပါ,ခရီးကောင်းပါ
4,10,𓂨𓀀𓎡 𓈙𓉔𓂝𓂋𓂋 𓃀𓐍𓐍𓇌,နင့်နံ့က ကောင်းတယ်,မင်းက ကောင်းကောင်း ခံစားရတယ်။
5,20,𓈎𓊪𓆑𓈙 𓆑𓈙 𓊪𓎡𓃀𓂝,ဒါက ကြီးမားပါတယ်။,ဒါက ခွေးတစ်ကောင်ပါ။
6,40,𓆑𓈎 𓏏𓄿𓈙𓎛𓈎 𓂋𓎡𓏭𓈖,ကံမကောင်းခဲ့ဘူး။,အဲဒါက ငါ့ဟာ မဟုတ်ခဲ့ဘူး။
7,80,𓊪𓂝 𓅲𓀀𓎡𓎛𓇌 𓆑𓈎,သူတွေ့တယ်။,သူအောင်မြင်ခဲ့တယ်။
8,160,𓆑𓈎 𓃀𓂝𓈎𓈙 𓏏𓐍𓋴𓈙𓂝,ပိုဆိုးလာတယ်,ပိုကောင်းလာမယ်။
9,320,𓆑𓂋𓂋 𓏏𓐍𓋴𓈖 𓊪𓄿𓋴𓇌,ကျွန်တော် အလုပ်ကြိုးစားတယ်။,သင်ဟာ ကြိုးစားလုပ်ကိုင်တယ်။


In [40]:
import pandas as pd

print("1. Burmese -> Hieroglyph Translation Pipeline")

data = {
    "Metric / Aspect": [
        "Overall BLEU Score",
        "1-gram Precision",
        "2-gram Precision",
        "3-gram Precision",
        "4-gram Precision",
        "Brevity Penalty (BP)",
        "Length Ratio",
        "Token Lengths (Hyp / Ref)"
    ],
    "RNN Model": [
        "11.8",
        "39.3%",
        "17.7%",
        "7.7%",
        "3.7%",
        "1.000",
        "1.067",
        "1,678 / 1,573"
    ],
    "Transformer Model": [
        "0.0 (Diverged/Collapsed)",
        "0.0%",
        "0.0%",
        "0.0%",
        "0.0%",
        "0.117",
        "0.318",
        "500 / 1,573"
    ],
    "Ensemble Model": [
        "19.1",
        "32.0%",
        "15.7%",
        "11.4%",
        "23.1%",
        "1.000",
        "1.044",
        "998 / 956"
    ]
}

df = pd.DataFrame(data)
display(df)

1. Burmese -> Hieroglyph Translation Pipeline


,Metric / Aspect,RNN Model,Transformer Model,Ensemble Model
0,Overall BLEU Score,11.8,0.0 (Diverged/Collapsed),19.1
1,1-gram Precision,39.3%,0.0%,32.0%
2,2-gram Precision,17.7%,0.0%,15.7%
3,3-gram Precision,7.7%,0.0%,11.4%
4,4-gram Precision,3.7%,0.0%,23.1%
5,Brevity Penalty (BP),1.000,0.117,1.000
6,Length Ratio,1.067,0.318,1.044
7,Token Lengths (Hyp / Ref),"1,678 / 1,573","500 / 1,573",998 / 956


In [41]:
import pandas as pd

print("2. Hieroglyph -> Burmese Translation Pipeline")

data = {
    "Metric / Aspect": [
        "Overall BLEU Score",
        "1-gram Precision",
        "2-gram Precision",
        "3-gram Precision",
        "4-gram Precision",
        "Brevity Penalty (BP)",
        "Length Ratio",
        "Token Lengths (Hyp / Ref)"
    ],
    "RNN Model": [
        "19.8",
        "27.7%",
        "13.6%",
        "12.4%",
        "33.3%",
        "1.000",
        "1.024",
        "979 / 956"
    ],
    "Transformer Model": [
        "15.0",
        "31.0%",
        "14.6%",
        "9.1%",
        "12.5%",
        "1.000",
        "1.026",
        "981 / 956"
    ],
    "Ensemble Model": [
        "17.2",
        "45.4%",
        "21.8%",
        "11.3%",
        "7.8%",
        "1.000",
        "1.041",
        "1,638 / 1,573"
    ]
}

df = pd.DataFrame(data)
display(df)

2. Hieroglyph -> Burmese Translation Pipeline


,Metric / Aspect,RNN Model,Transformer Model,Ensemble Model
0,Overall BLEU Score,19.8,15.0,17.2
1,1-gram Precision,27.7%,31.0%,45.4%
2,2-gram Precision,13.6%,14.6%,21.8%
3,3-gram Precision,12.4%,9.1%,11.3%
4,4-gram Precision,33.3%,12.5%,7.8%
5,Brevity Penalty (BP),1.000,1.000,1.000
6,Length Ratio,1.024,1.026,1.041
7,Token Lengths (Hyp / Ref),979 / 956,981 / 956,"1,638 / 1,573"
